# Predicting 30-Day Hospital Readmission in Patients with Diabetes

## 1. Problem Definition

### Objective
Predict whether a hospital encounter involving a patient with diabetes
will be followed by a recorded readmission within 30 days, using
information available at discharge.

### Intended Use
Explore whether a prediction model could help care teams prioritize
patients for post-discharge follow-up.

### Prediction Target
- Positive class (1): `readmitted` is `<30`.
- Negative class (0): `readmitted` is `>30` or `NO`.

### Unit of Observation
Each row represents a hospital encounter. A patient can appear in
multiple encounters, so patients must be kept separate across
training, validation, and test sets.

### Scope
This is an educational prototype using historical data.
Its performance does not establish suitability for clinical use.

## 2. Data Loading and Initial Inspection

The dataset contains hospital encounters involving patients with diabetes
from 130 US hospitals and integrated delivery networks between 1999 and 2008.

Source: https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

Before cleaning, we inspect the dataset's dimensions, column types,
missing-value markers, patient identifiers, and target distribution.

In [3]:
import pandas as pd

In [17]:
df_raw = pd.read_csv("data/raw/diabetic_data.csv",keep_default_na=False, na_values=["?", ""])

C:\Users\stany\AppData\Local\Temp\ipykernel_13584\453683866.py:1: DtypeWarning: Columns (10: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("data/raw/diabetic_data.csv",keep_default_na=False, na_values=["?", ""])


In [18]:
df_raw.shape

(101766, 50)

In [19]:
df_raw.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

In [ ]:
df_raw.nunique()

encounter_id                101766
patient_nbr                  71518
race                             5
gender                           3
age                             10
weight                           9
admission_type_id                8
discharge_disposition_id        26
admission_source_id             17
time_in_hospital                14
payer_code                      17
medical_specialty               72
num_lab_procedures             118
num_procedures                   7
num_medications                 75
number_outpatient               39
number_emergency                33
number_inpatient                21
diag_1                         716
diag_2                         748
diag_3                         789
number_diagnoses                16
max_glu_serum                    4
A1Cresult                        4
metformin                        4
repaglinide                      4
nateglinide                      4
chlorpropamide                   4
glimepiride         

In [21]:
df_raw.isnull().sum()

encounter_id                    0
patient_nbr                     0
race                         2273
gender                          0
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty           49949
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum                   0
A1Cresult                       0
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

In [22]:
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": df_raw.isna().mean() * 100
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_pct", ascending=False)
)

missing_summary.round(2)

,missing_count,missing_pct
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


In [34]:
df = df_raw.copy()
columns_to_fill = ["medical_specialty",
    "payer_code",
    "race",
    "diag_1",
    "diag_2",
    "diag_3"]

df[columns_to_fill] = df[columns_to_fill].fillna("unknown")
df = df.drop(columns=["weight"]) #removing weight as we are missing 96.86% of the val

df[columns_to_fill].isnull().sum()

medical_specialty    0
payer_code           0
race                 0
diag_1               0
diag_2               0
diag_3               0
dtype: int64

In [35]:
test_columns = ["max_glu_serum", "A1Cresult"]
df[test_columns] = df[test_columns].replace("None", "not_measured")

### Missing-value handling

Categorical missing values were replaced with an explicit "unknown"
category to retain encounters without inventing information.

Weight was excluded from the initial model inputs because 96.86% of
values were missing.

For glucose and HbA1c test results, "None" was renamed "not_measured"
to preserve its documented meaning.

### Patient and Encounter Identifiers

Each row represents a hospital encounter, while a patient may have
multiple encounters.

We inspect encounter uniqueness and repeated patient identifiers.
All encounters belonging to the same patient will be assigned to
the same data split to evaluate generalization to unseen patients.

Identifiers will not be used as predictive features.

In [43]:
print(f"Total encounter: {len(df)}")
print(f"# unique encounterid:{df["encounter_id"].nunique()}")
print(f"# unique patient: {df["patient_nbr"].nunique()}")

Total encounter: 101766
# unique encounterid:101766
# unique patient: 71518


In [40]:
len(df["encounter_id"])

101766

In [45]:
df["readmitted"].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [53]:
readmitted_summary = pd.DataFrame({
    'count': round(df["readmitted"].value_counts(), 2),
    'pct': round(df["readmitted"].value_counts(normalize=True)*100, 2)
})

In [54]:
readmitted_summary

,count,pct
readmitted,,
NO,54864,53.91
>30,35545,34.93
<30,11357,11.16


Readmissions within 30 days account for 11.16% of hospital encounters, making the positive class a minority. A model that always predicts no readmission within 30 days would achieve 88.84% accuracy, while failing to identify any early readmissions. Accuracy alone is therefore insufficient, therefore precision and recall will also be used to assess model performance.

## 3. Target Definition and Data Splitting

### 3.1 Binary Target Definition

The target `readmitted_30d` is set to 1 for readmissions within
30 days (`<30`) and 0 otherwise (`>30` or `NO`).
The original `readmitted` column is preserved for reference.

In [56]:
df["readmitted_30d"] = (df["readmitted"]== "<30").astype(int)

In [63]:
# Check that the positive target count matches the original "<30" count
(df["readmitted_30d"]==1).sum()


np.int64(11357)

### 3.2 Patient-Level Data Splitting

Patients are split into training, validation, and test sets.
All encounters belonging to the same patient remain in the same set.


In [65]:
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, 
                             test_size=0.20,
                             random_state=2)
full_train_pos, test_pos = next(splitter.split(df, groups=df["patient_nbr"]))

df_full_train = df.iloc[full_train_pos].copy()
df_test = df.iloc[test_pos].copy()

In [68]:
val_splitter = GroupShuffleSplit(n_splits=1,
                                 test_size=0.25,
                                 random_state=2)
train_pos, val_pos = next(val_splitter.split(df, groups=df["patient_nbr"]))
df_train = df.iloc[train_pos].copy()
df_val = df.iloc[test_pos].copy()